# Tutorial 4: Evaluating LLM Agents on Mathematical Reasoning

Welcome to the fourth tutorial in our AI Safety Evaluations course.

So far you have evaluated models as **passive responders** — the model reads a prompt,
produces an answer, and you score it. But many real-world AI systems are **agents**: they
observe, reason, act (e.g. call a tool), observe the result, and repeat. Evaluating agents
is harder because the model's behaviour is no longer a single forward pass — it is a
*multi-step trajectory* where mistakes compound and new failure modes (infinite loops,
tool misuse, hallucinated tool calls) appear.

In this tutorial you will build and evaluate a simple **ReAct agent** that solves
math problems by calling calculator and algebra tools. You will see first-hand how
scaffolding choices — prompts, tool sets, message limits, output formatting — affect
agent performance, and you will practice a basic dev/test workflow for iterating on
an agent without overfitting to your evaluation set. Think of it as a toy,
simplified version of a real elicitation pipeline like the
[METR Elicitation Protocol](https://evaluations.metr.org/elicitation-protocol/).

**What you'll learn:**

- Define custom tools for inspect_ai agents
- Build a ReAct agent and iteratively improve it on a dev set
- Develop intuition for how scaffolding choices affect agent performance

**By the end:** **You'll have a working agent evaluation pipeline and hands-on experience with the kind of iteration loop used in real-world agent evals.**

## 1. Setup

**Model choice.** We recommend picking a model that isn't too powerful — ideally one that occasionally
stumbles on arithmetic so you can observe the effect of giving it tools. The examples
below use `qwen2.5:3b`, but feel free to swap in any model you have access to (just make sure it supports tool calling). The
main goal is to see how well the model uses the tools it's given, so don't worry if
the tool-augmented score ends up lower than plain generation — that's a valid and
interesting finding, not a sign that something is broken. Conversely, if your model
solves everything perfectly even without tools, consider switching to a harder dataset
(e.g. the full MATH instead of MATH-500, or a competition-math set like AIME) — just
make sure to note this in your write-up.

> **Resource note:** Agent evaluations generate many more LLM calls than simple Q&A evals
> because each problem may involve multiple reasoning steps. All `eval()` calls in this
> notebook use a `limit` parameter to cap the number of samples processed. Adjust
> `EVAL_LIMIT` and `MAX_MESSAGES` if your machine is slow.

In [ ]:
!pip install sympy datasets scipy -q
print("✅ Installed: sympy, datasets, scipy")

In [50]:
import re
import math
import random
from textwrap import dedent
from collections import defaultdict

from scipy.stats import norm

from inspect_ai import Task, eval
from inspect_ai.dataset import Sample, hf_dataset
from inspect_ai.agent import react, AgentSubmit
from inspect_ai.solver import generate, use_tools, system_message, TaskState
from inspect_ai.scorer import (
    Score, Target, model_graded_qa, scorer, accuracy, stderr, match, mean
)
from inspect_ai.tool import tool

from dotenv import load_dotenv
from sympy import solve, sympify
from sympy.parsing.sympy_parser import standard_transformations, implicit_multiplication_application

_transformations = (standard_transformations +
    (implicit_multiplication_application,))

load_dotenv()

True

In [5]:
# Configure model -- replace with what is available in your environment.
# E.g. 'ollama/qwen2.5:7b', 'openai/gpt-4o-mini', 'anthropic/claude-haiku-4-5'
# Must support tool calling (look for "tools" tag in Ollama).

MODEL = "ollama/qwen2.5:3b"

RANDOM_SEED = 42
EVAL_LIMIT = 30        # max samples per eval run (raise if your machine is fast)
MAX_MESSAGES = 20      # max back-and-forth messages per agent trajectory

A few helper functions for extracting and displaying results. You don't need to modify
these — just run the cell and move on.

In [6]:
def get_acc(log):
    """Extract accuracy (or mean) from the first scorer in a log."""
    m = log.results.scores[0].metrics
    return (m.get("accuracy") or m.get("mean")).value


def _first_score(sample):
    """Get the first Score object from a sample, regardless of scorer name."""
    scores = sample.scores
    first_key = list(scores.keys())[0]
    val = scores[first_key]
    return val[0] if isinstance(val, list) else val


def print_results(label, log):
    """Pretty-print per-sample results from an eval log."""
    acc = get_acc(log)
    print(f"{'=' * 60}")
    print(f"  {label}   accuracy: {acc:.0%}")
    print(f"{'=' * 60}")
    for i, sample in enumerate(log.samples, 1):
        sc = _first_score(sample)
        msgs = len(sample.messages)
        expl = (sc.explanation or "")[:60]
        print(f"  [{sc.value}] #{i:2d}  msgs={msgs:2d}  "
              f"target={sample.target[:20]:>20s}  {expl}")
    print()

## 2. Tools and Agent Architecture

### Why agents?

A standard LLM evaluation looks like this: you hand the model a question, it produces
an answer, and a scorer checks whether the answer is correct. The model has *one shot*.

But many practical AI systems need to **interact with the world** — search the web, run
code, query a database, or call an API — before they can answer. These systems are
called **agents**. An agent follows a loop:

1. **Observe** the current state (the question, plus any tool outputs so far).
2. **Think** about what to do next.
3. **Act** by calling a tool (or submitting a final answer).
4. **Observe** the tool's result, then go back to step 2.

This loop continues until the agent decides it has enough information to submit a final
answer, or until a safety limit (maximum number of steps) is reached.

### Why not just give the model tools and let it figure things out?

Because *access* to tools is not enough. The model also needs:

- A **system prompt** that tells it what tools exist and how to use them
- A **loop structure** that feeds tool results back so the model can reason about them
- A **stopping criterion** so the model knows when and how to submit its final answer
- **Message limits** to prevent runaway loops that burn tokens without progress

This combination of prompt + loop + stopping logic is called **scaffolding**, and it
can make or break agent performance. Whether an agent actually helps depends on the
task, the tools, the prompt, and the model — scaffolding is not a guaranteed win, but
understanding it is essential for evaluating agent systems.

### Approaches to giving a model tools

The simplest way to add tools in inspect_ai is the **`use_tools()` + `generate()`** pattern.
`use_tools()` registers a list of tool functions so the model can call them, and
`generate()` runs a **single generation**. The model may call tools during that generation,
but the scaffolding never interrupts — tool calls and the final answer are produced in
one continuous flow, with no structured pause to reconsider.
```python
solver = [
    system_message("You have access to calculator tools."),
    use_tools([add(), multiply()]),
    generate(),
]
```

This is easy to set up but fragile: if a tool returns something unexpected mid-generation,
the model is already committed to a line of reasoning and may not change course.

### The ReAct pattern

**ReAct** (Reason + Act) is a scaffolding pattern that introduces an explicit pause after
every tool call. The model reasons and calls one tool; the scaffolding then appends the
result to the context and starts a **fresh generation** — so the model reconsiders its plan
from scratch before deciding the next step. This closed feedback loop makes it much easier
to recover from unexpected tool results or multi-step reasoning chains.
```
┌─────────────────────────────────────────────────────────┐
│                    ReAct Agent Loop                      │
│                                                         │
│   ┌──────────┐    ┌──────────┐    ┌──────────┐         │
│   │  THINK   │───>│   ACT    │───>│ OBSERVE  │──┐      │
│   │          │    │          │    │          │  │      │
│   │ "I need  │    │ call     │    │ tool     │  │      │
│   │  to add  │    │ add(a,b) │    │ returns  │  │      │
│   │  these"  │    │          │    │ "111915" │  │      │
│   └──────────┘    └──────────┘    └──────────┘  │      │
│        ^                                        │      │
│        └────────────────────────────────────────┘      │
│                                                         │
│   Loop continues until the agent calls submit()         │
│   or the message limit is reached.                      │
└─────────────────────────────────────────────────────────┘
```

In **inspect_ai**, the `react()` solver implements this pattern. It:
1. Sends the problem with a system prompt describing available tools
2. Lets the model think and call one tool at a time
3. Appends the tool result to the context and starts a fresh generation
4. Repeats until the model calls `submit()` (a built-in action added automatically)

The `submit()` tool is special — it signals the end of the loop and its argument
becomes the agent's final answer. You never define `submit()` yourself; `react()`
adds it for you.

### Defining tools in inspect_ai

An agent needs *actions* it can take in the world. In inspect_ai, actions are
**tools** — Python functions the model can call during its reasoning loop.

The `@tool` decorator registers a function so that inspect_ai can:
1. **Describe** it to the model (via the docstring and type hints — these become the
   tool schema the model sees).
2. **Execute** it when the model emits a tool-call message and return the result.

The pattern is a factory function (decorated with `@tool`) that returns an `async`
inner function. The inner function's docstring and parameter annotations are what the
model actually sees, so clear names and descriptions directly affect agent performance.

Below we define four arithmetic tools ourselves. Notice how each one:
- Takes typed parameters with descriptive `Args:` docstrings
- Returns a **string** (tool outputs are always strings in inspect_ai)
- Wraps execution in `try/except` so the agent gets a readable error instead of a crash

> **Built-in tools:** inspect_ai also ships with ready-made tools for common agent tasks — you don't need to write these yourself:
> - `bash()` — run shell commands
> - `python()` — execute Python code
> - `web_search()` — search the web
> - `web_browser()` — full browser interaction
> - `text_editor()` — read and edit files
>
> For the full list see the [Tools section of the inspect_ai documentation](https://inspect.ai-safety-institute.org.uk/tools.html).
> In this tutorial we write our own tools from scratch to understand the mechanics,
> but in practice you will often mix custom tools with these built-in ones.

In [10]:
@tool
def add():
    async def execute(a: float, b: float) -> str:
        """Add two numbers.

        Args:
            a: First number.
            b: Second number.
        """
        try:
            return str(float(a) + float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def subtract():
    async def execute(a: float, b: float) -> str:
        """Subtract b from a.

        Args:
            a: Number to subtract from.
            b: Number to subtract.
        """
        try:
            return str(float(a) - float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def multiply():
    async def execute(a: float, b: float) -> str:
        """Multiply two numbers.

        Args:
            a: First number.
            b: Second number.
        """
        try:
            return str(float(a) * float(b))
        except Exception as e:
            return f"Error: {e}"
    return execute


@tool
def divide():
    async def execute(a: float, b: float) -> str:
        """Divide a by b.

        Args:
            a: Dividend.
            b: Divisor (must not be zero).
        """
        try:
            b_val = float(b)
            if b_val == 0:
                return "Error: division by zero."
            return str(float(a) / b_val)
        except Exception as e:
            return f"Error: {e}"
    return execute

## Assignment 1: Create a `modular_arithmetic` tool

Cryptography and number theory problems often require modular arithmetic —
for example, computing $7^{1000} \mod 13$ as part of a larger proof.

Following the pattern above, implement a `modular_arithmetic` tool with a clear docstring.

In [11]:
@tool
def modular_arithmetic():
    async def execute(a: int, b: int) -> str:
        """Compute a mod b. Example, modular_arithmetic(a: 11, b: 3) is 2.
        
        Args:
            a: Modular dividend.
            b: Modular divisor (must not be zero).
        """
        try:
            b_val = int(b)
            if b_val == 0:
                return "Error: division by zero."
            a_val = int(a)
            return str(a_val % b_val)
        except Exception as e:
            return f"Error: {e}"
        
    return execute

Now let's see the tool in action. We run a small eval so the model attempts to use your tool. Don't worry if the model gets the answer wrong — what matters here is that the tool itself is correctly defined.

In [24]:
_test_samples_mod = [
    Sample(input="What is 1000000 mod 397? Reply with just the number.", target="354"),
    Sample(input="What is 100 mod 10? Reply with just the number.", target="0"),
]

_log_mod_test = eval(
    Task(
        dataset=_test_samples_mod,
        solver=react(
            prompt="""
            You have a `modular_arithmetic` tool. Always use it to calculate answer.
            """,
            tools=[modular_arithmetic()],
            attempts=1,
            submit=AgentSubmit(answer_only=True, description="""Answer the task. Example, `submit(answer: "1234")`
            Args:
                answer: string with the answer
            """),
        ),
        scorer=match(numeric=True, location="exact"),  # !fixed to exact
        message_limit=10,
        epochs=5,
    ),
    model=MODEL,
)[0]

print_results("modular_arithmetic tool test", _log_mod_test)

Output()

  modular_arithmetic tool test   accuracy: 70%
  [C] # 1  msgs= 7  target=                 354  354
  [I] # 2  msgs= 5  target=                   0  10
  [C] # 3  msgs= 5  target=                 354  354
  [C] # 4  msgs= 9  target=                   0  0
  [I] # 5  msgs=10  target=                 354   submit(answer: "354")
  [I] # 6  msgs= 5  target=                   0  10
  [C] # 7  msgs= 7  target=                 354  354
  [C] # 8  msgs= 9  target=                   0  0
  [C] # 9  msgs= 7  target=                 354  354
  [C] #10  msgs= 7  target=                   0  0



> **Checking tool usage.** Run **`inspect view`** in the terminal to see the full
> message trace for each sample — tool calls and their responses appear as explicit
> steps. Alternatively, inspect `log.samples[i].messages` directly: each tool
> call appears as a message with `role="assistant"` and a non-empty **`tool_calls`** field.

---
1. Did the model actually *use* your tool, or did it answer without using it?
   Open the eval log in the inspect_ai viewer (`inspect view`) and check the trace —
   you should see explicit tool call steps between the initial question and the final answer.
2. If the model skipped the tool, adjust the prompt in the `react()` call above and
   re-run until the model uses it. What did you change?


## answer
  - modular_arithmetic tool test   accuracy: 70%
  - [C] # 1  msgs= 7  target=                 354  354
  - [I] # 2  msgs= 5  target=                   0  10
  - [C] # 3  msgs= 5  target=                 354  354
  - [C] # 4  msgs= 9  target=                   0  0
  - [I] # 5  msgs=10  target=                 354   submit(answer: "354")
  - [I] # 6  msgs= 5  target=                   0  10
  - [C] # 7  msgs= 7  target=                 354  354
  - [C] # 8  msgs= 9  target=                   0  0
  - [C] # 9  msgs= 7  target=                 354  354
  - [C] #10  msgs= 7  target=                   0  0

1. Yes in 1st case, no in the second case (it just ignored the modular tool and ran `submit` tool).
2. I changed prompt (I added "always use tool"), description for modular tool, and I changed prompt for submit tool.

In [12]:
ARITH_TOOLS = [add(), subtract(), multiply(), divide(), modular_arithmetic()]

## 3. Toy Evaluation — Three Solver Architectures

Before touching a real benchmark, let's try out three different solver architectures
on a small set of hand-crafted problems. These 12 word problems use numbers large
enough that a model without tools might make arithmetic errors.

We will compare three solver architectures:

- **Plain generation** - reads the question, produces an answer in one shot;
  solver: `generate()`
- **Naive tool loop** - gets access to tools; `generate()` runs once and may call
  some tools; solver: `use_tools()` + `generate()`
- **ReAct agent** - explicit think-act-observe loop with a `submit()` action to stop;
  solver: `react()`
  
The goal here is to see how each architecture behaves — both in terms of accuracy
and how the solver actually runs — before moving to a real benchmark.

In [25]:
TOY_SAMPLES = [
    Sample(
        input=(
            "A semiconductor factory produced 48,397 chips on Monday "
            "and 63,518 chips on Tuesday. How many chips were produced in total?"
        ),
        target="111915",
    ),
    Sample(
        input=(
            "A government reserve had 874,203 barrels of oil. "
            "After an emergency release, 295,867 barrels were distributed. "
            "How many barrels remain in the reserve?"
        ),
        target="578336",
    ),
    Sample(
        input=(
            "A logistics company ships 4,738 containers, each holding 2,659 units. "
            "How many units are shipped in total?"
        ),
        target="12598342",
    ),
    Sample(
        input=(
            "A national census counted 8,743,291 residents across 6,473 districts. "
            "If residents are distributed equally, how many full residents "
            "are assigned per district?"
        ),
        target="1350",
    ),
    Sample(
        input=(
            "A satellite completes a full orbit every 397 minutes. "
            "After exactly 1,000,000 minutes of operation, how many minutes "
            "have passed since the last complete orbit?"
        ),
        target="354",
    ),
    Sample(
        input=(
            "A hospital ordered 12,475 boxes of supplies at 387 dollars per box. "
            "They received a bulk discount of 843,750 dollars off the total. "
            "How much did the hospital pay after the discount?"
        ),
        target="3984075",
    ),
    Sample(
        input=(
            "A city has 14 times as many residents as municipal employees. "
            "If the total number of residents and employees together is 489,375, "
            "how many municipal employees does the city have?"
        ),
        target="32625",
    ),
    Sample(
        input=(
            "An airline flew 3,847 domestic flights and 2,964 international flights "
            "last month. Each flight used an average of 8,753 liters of fuel. "
            "How many liters of fuel were used in total?"
        ),
        target="59616683",
    ),
    Sample(
        input=(
            "A clock tower rings a bell every 1,873 seconds. "
            "After 10,000,000 seconds have elapsed since midnight, "
            "how many seconds ago did the bell last ring?"
        ),
        target="53",
    ),
    Sample(
        input=(
            "A farm harvested 247,839 kg of wheat and 184,672 kg of barley. "
            "The grain is loaded into trucks that carry exactly 4,750 kg each. "
            "How many full truckloads can be made from all the grain?"
        ),
        target="91",
    ),
    Sample(
        input=(
            "A global streaming platform has 1,847,293,847,291 seconds of video content. "
            "Given that a day has 86,400 seconds, how many full days of content "
            "does the platform have?"
        ),
        target="21380715",
    ),
    Sample(
        input=(
            "A country's economy grew by 3,847 dollars per citizen in a year. "
            "The country has 847,293,847 citizens. "
            "What was the total economic growth in dollars?"
        ),
        target="3259539429409",
    ),
]

## 3a. Approach 0: Plain generation (no tools)

The simplest baseline: give the model a system prompt and ask it to solve the problem
directly. The solver is just `generate()` — a single forward pass with no tool access.

We score with `match(numeric=True)`, which extracts the first number from the model's
response and compares it to the target.

In [26]:
SIMPLE_PROMPT = dedent("""    You are a math solver. Read the problem carefully, compute the answer,
    and respond with the final numeric result.
""")

log_toy_gen = eval(
    Task(
        dataset=TOY_SAMPLES,
        solver=[system_message(SIMPLE_PROMPT), generate()],
        scorer=match(numeric=True),
    ),
    model=MODEL,
)[0]

print_results("Approach 0: generate() only", log_toy_gen)

Output()

  Approach 0: generate() only   accuracy: 33%
  [C] # 1  msgs= 3  target=              111915  To find the total number of chips produced by the factory ov
  [C] # 2  msgs= 3  target=              578336  To find out how many barrels remain in the reserve after the
  [I] # 3  msgs= 3  target=            12598342  To find the total number of units shipped, multiply the numb
  [I] # 4  msgs= 3  target=                1350  To find out how many full residents are assigned per distric
  [I] # 5  msgs= 3  target=                 354  To find out how many minutes have passed since the last comp
  [I] # 6  msgs= 3  target=             3984075  To calculate how much the hospital paid after the discount f
  [C] # 7  msgs= 3  target=               32625  Let's denote the number of municipal employees by \( E \). T
  [I] # 8  msgs= 3  target=            59616683  To solve this problem, first we need to find out the total n
  [I] # 9  msgs= 3  target=                  53  To determine how many sec

---
1. Did the model get everything right, or did it make arithmetic errors on the larger numbers?
2. If there were errors, what do you think caused them?

**Your answer:**
1. It makes errors with big numbers or division. 
2. It can add or multiply better then divide. Also sometimes it fails to copy answer string - it can even skip or add digit.

## 3b. Approach A: Naive tool loop

Now we give the model access to our arithmetic tools via `use_tools()`, followed by a
single `generate()`. The model *can* call tools, but the solver doesn't enforce any
structure: it may call one tool, multiple tools, or none at all, and it generates a
final answer in the same pass.

> In practice this pattern is rarely used on its own — without scaffolding, whether the
> model actually uses the tools is largely unpredictable.

In [29]:
NAIVE_LOOP_PROMPT = dedent("""    You are a math solver with access to calculator tools.
    Break each problem into arithmetic steps and call one tool per step.
""")

log_toy_naive = eval(
    Task(
        dataset=TOY_SAMPLES,
        solver=[
            system_message(NAIVE_LOOP_PROMPT),
            use_tools(ARITH_TOOLS),
            generate(),
        ],
        scorer=match(numeric=True),
    ),
    model=MODEL,
)[0]

print_results("Approach A: use_tools + generate (naive loop)", log_toy_naive)

Output()

  Approach A: use_tools + generate (naive loop)   accuracy: 25%
  [I] # 1  msgs= 3  target=              111915  
  [I] # 2  msgs= 3  target=              578336  
  [C] # 3  msgs= 7  target=            12598342  The total number of units shipped is 12,598,342.

To find th
  [I] # 4  msgs= 3  target=                1350  
  [I] # 5  msgs= 7  target=                 354  We have calculated that after 1,000,000 minutes of operation
  [C] # 6  msgs= 5  target=             3984075  The initial total cost for 12,475 boxes at $387 per box is c
  [I] # 7  msgs= 3  target=               32625  
  [C] # 8  msgs= 7  target=            59616683  The total liters of fuel used is \( 59,616,683 \).

Therefor
  [I] # 9  msgs= 5  target=                  53  After performing the calculation, we find that it has been 9
  [I] #10  msgs= 7  target=                  91  Dividing the total harvest by the capacity of each truck giv
  [I] #11  msgs= 3  target=            21380715  
  [I] #12  msgs= 3  target

---
1. Did having access to tools improve results compared to the baseline?
2. Did the model actually use the tools, or did it ignore them?

**Your answer:**
1. 33% on baseline, 25% with naive tools. **No.**. Sometimes (>1) it fails to produce any tokens.
2. Sometimes it ignores tools and try to compute on it's own.

## 3c. Approach B: ReAct agent

Now let's use the `react()` solver — the full ReAct loop described in the introduction.
Notice the prompt explicitly tells the model to use tools and call `submit()` at the end.

In [30]:
REACT_PROMPT_V1 = dedent("""    You are a math solver with access to calculator tools.
    Break each problem into arithmetic steps and call one tool per step.
    Don't calculate anything without tools.
    After getting the final numeric result, call submit() with ONLY the number.
""")

log_toy_react = eval(
    Task(
        dataset=TOY_SAMPLES,
        solver=react(prompt=REACT_PROMPT_V1, tools=ARITH_TOOLS, attempts=1),
        scorer=match(numeric=True),
        message_limit=20,
    ),
    model=MODEL,
)[0]

print_results("Approach B: react() with simple prompt", log_toy_react)

Output()

[04/08/26 00:56:03] ERROR    Exception in callback Task.__step()                                ]8;id=692855;file:///home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/base_events.py\base_events.py]8;;\:]8;id=964757;file:///home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/base_events.py#1765\1765]8;;\
                             handle: <Handle Task.__step()>                                                        
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "/home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/events.p                    
                             y", line 80, in _run                                                                  
                                 self._context.run(self._callback, *self._args)                                    
                             RuntimeError: cannot enter context: <_contextvars.Context object                      
                             at 0x7ffb281aefc0> is already entered                                                 

  Approach B: react() with simple prompt   accuracy: 83%
  [C] # 1  msgs= 7  target=              111915  111915.0
  [C] # 2  msgs=11  target=              578336  It seems like there's a persistent issue with passing the nu
  [C] # 3  msgs= 5  target=            12598342  The total number of units shipped is calculated as 4738 * 26
  [C] # 4  msgs=10  target=                1350  1350
  [C] # 5  msgs= 5  target=                 354  The calculation shows that after exactly 1,000,000 minutes o
  [I] # 6  msgs=20  target=             3984075  
  [C] # 7  msgs= 8  target=               32625  The city has 32,625 municipal employees.

I will now call th
  [C] # 8  msgs=15  target=            59616683  The total liters of fuel used by the airline for all flights
  [I] # 9  msgs=18  target=                  53  9.313225746154785e-09
  [C] #10  msgs=13  target=                  91  It seems there's a format issue with submitting the integer 
  [C] #11  msgs=17  target=            21380715  2

## Comparing the three approaches

Let's see the results side by side. Pay attention not just to accuracy but also to the
number of messages — more messages means more LLM calls, which means more cost and latency.

In [31]:
TOY_LOGS = [log_toy_gen, log_toy_naive, log_toy_react]
TOY_LABELS = ["generate only", "naive tool loop", "react v1"]

print(f"{'Approach':<25s} {'Acc':>5s}  {'Avg msgs':>8s}  {'Max msgs':>8s}")
print("-" * 50)
for label, log in zip(TOY_LABELS, TOY_LOGS):
    acc = get_acc(log)
    msg_list = [len(s.messages) for s in log.samples]
    avg_m = sum(msg_list) / len(msg_list)
    max_m = max(msg_list)
    print(f"{label:<25s} {acc:>4.0%}   {avg_m:>7.1f}   {max_m:>7d}")

Approach                    Acc  Avg msgs  Max msgs
--------------------------------------------------
generate only              33%       3.0         3
naive tool loop            25%       4.7         7
react v1                   83%      11.2        20


---
1. Which approach performed best? Was it also the most expensive (most messages)?
2. Open the eval and compare the naive tool loop and the ReAct
   agent traces. Did the model actually use the tools in both cases, and how does the
   structure of the traces differ?

**Your answer:**

1. react v1 is the most accurate and the most expensive.
2. React v1 use more tools (and messages). At first he write a plan, then next step he make tool calls one-by-one. Naive realisation starts tool calling in the first message.

## 4. Adding a Symbolic Algebra Tool

Arithmetic tools help with computation, but many math problems require solving
equations — "find x such that 3x + 7 = 22". A small model can't do this reliably
without tools, but SymPy (a Python symbolic math library) can solve it exactly.

## Assignment 2: Create the `sympy_solve` tool

Implement a tool that takes an equation string (e.g. `"2*x + 5 = 21"` or `"3*x**2 - 12 = 0"`)
and returns the solutions using SymPy. The tool should:

1. Parse the equation — if it contains `=`, split into left and right sides and solve `left - right = 0`
2. If there's no `=`, treat the input as an expression equal to zero
3. Solve for the symbol `x`
4. Return the solutions as a string, or `"No solution found."` if empty
5. Handle errors gracefully

In [14]:
@tool
def sympy_solve():
    async def execute(equation: str) -> str:
        '''Solve the equation for x.
        Args:
            equation: equation for x.

        Returns:
            comma separated solutions

        Examples:
            >>> execute('2x=4')
            2
            >>> execute('x**2-5*x+3=3')
            2, 3
        '''
        if not isinstance(equation, str):
            return "Error: equation is not string."
        eqs = equation.split('=')
        if len(eqs) <= 1:
            return "Error: equation must contain exactly 1 equal sign"
        elif len(eqs) >= 3:
            return "Error: equation must contain exactly 1 equal sign"
        try:
            eq0_sympy = parse_expr(eqs[0], transformations=_transformations)
        except Exception:
            return 'Error: incorrect equation'

        try:
            eq1_sympy = parse_expr(eqs[1], transformations=_transformations)
        except Exception:
            return 'Error: incorrect equation'
        
        result_sympy = eq0_sympy - eq1_sympy
        try:
            solutions = solve(result_sympy)
        except Exception:
            return "Error: can't solve"
        
        if not isinstance(solutions, list):
            return solutions
        elif len(solutions) == 0:
            return "No solutions."
        else:
            return ', '.join(str(s) for s in solutions)
            
    return execute

In [55]:
# Run the eval — the model should use your tool to solve both equations.
_log_sympy_test = eval(
    Task(
        dataset=[
            Sample(
                input="Solve for x: 2*x + 5 = 21. Reply with just the number.",
                target="8",
            ),
            Sample(
                input="Solve for x: x**2 - 9 = 0. What are the solutions? Reply with just the numbers separated by comma.",
                target="-3, 3",
            ),
        ],
        solver=react(
            prompt="You have a sympy_solve(equation) tool. Use it to solve the equation, then submit the result.",
            tools=[sympy_solve()],
            attempts=1,
        ),
        scorer=match(numeric=True),
        message_limit=10,
    ),
    model=MODEL,
)[0]

print_results("sympy_solve tool test", _log_sympy_test)

Output()

  sympy_solve tool test   accuracy: 100%
  [C] # 1  msgs= 7  target=                   8  8
  [C] # 2  msgs= 7  target=               -3, 3  -3, 3



---
1. Did the model use the `sympy_solve` tool, or did it try to solve the equations
   in its head? (Check the message counts.)
2. If it didn't use the tool, try adjusting the prompt in the `react()` call.
   What wording helped?

**Your answer:**
1. It uses sympy toolcall for both problems.
2. It works.

In [15]:
# Bundle all tools
ARITH_TOOLS = [add(), subtract(), multiply(), divide(), modular_arithmetic()]
ALL_TOOLS = ARITH_TOOLS + [sympy_solve()]

## 5. Loading the MATH-500 Benchmark

Now that we have a working agent with tools, let's evaluate it on a real benchmark.
**MATH-500** is a 500-question subset of the MATH dataset (Hendrycks et al., 2021),
covering competition-level math across seven subjects. It's available on Hugging Face.

Not all subjects benefit equally from our tools — Geometry and Counting & Probability
involve spatial reasoning and combinatorics that our calculator tools can't help with.
We'll focus on the four subjects where arithmetic and algebra tools are most relevant:
Algebra, Intermediate Algebra, Number Theory, and Prealgebra.

### Extracting answers from MATH solutions

MATH stores answers inside `\boxed{...}` in the solution string. We need a helper to
extract them:

In [16]:
TOOL_SUBJECTS = [
    "Algebra", "Number Theory", "Prealgebra", "Intermediate Algebra",
]


def extract_boxed(solution):
    """Extract the content of the last \\boxed{...} in a MATH solution string."""
    idx = solution.rfind("\\boxed{")
    if idx == -1:
        return solution.strip()
    start = idx + len("\\boxed{")
    depth = 1
    i = start
    while i < len(solution) and depth > 0:
        if solution[i] == "{":
            depth += 1
        elif solution[i] == "}":
            depth -= 1
        i += 1
    return solution[start:i - 1].strip()


def record_to_sample(record):
    """Convert a MATH-500 record into an inspect_ai Sample."""
    target = record.get("answer") or extract_boxed(record["solution"])
    return Sample(
        input=record["problem"],
        target=target,
        metadata={
            "level": int(record["level"]),
            "subject": record["subject"],
        },
    )

In [17]:
full_dataset = hf_dataset(
    path="HuggingFaceH4/MATH-500",
    split="test",
    sample_fields=record_to_sample,
    cached=True,
)

print(f"Total MATH-500: {len(full_dataset)} samples")

subject_counts = defaultdict(int)
for s in full_dataset:
    subject_counts[s.metadata["subject"]] += 1

print(f"\n{'Subject':<30s} {'Count':>5s}")
print("-" * 37)
for subj in sorted(subject_counts):
    marker = " <-- tool-friendly" if subj in TOOL_SUBJECTS else ""
    print(f"{subj:<30s} {subject_counts[subj]:>5d}{marker}")

Total MATH-500: 500 samples

Subject                        Count
-------------------------------------
Algebra                          124 <-- tool-friendly
Counting & Probability            38
Geometry                          41
Intermediate Algebra              97 <-- tool-friendly
Number Theory                     62 <-- tool-friendly
Prealgebra                        82 <-- tool-friendly
Precalculus                       56


## Dev/test split

A core principle in evaluation: **never tune on your test set.** Iterating on the same
data you use for final scoring inflates results and makes them meaningless. We split the
tool-friendly subset into a small **dev set** (10%) for prompt and scaffolding iteration,
and a larger **test set** (90%) reserved for final evaluation only.

This mirrors the elicitation workflow described in [METR's Guidelines for Capability Elicitation](https://evaluations.metr.org/elicitation-protocol/): iterate against the dev set until failures stabilize, then run the test set once.

> **Adjust to your hardware.** If even the dev set takes too long, reduce `split_point`
> or set a smaller `EVAL_LIMIT`. The point is rapid iteration — you can always increase
> the test set size later.

In [18]:
tool_dataset = [s for s in full_dataset if s.metadata["subject"] in TOOL_SUBJECTS]
print(f"Tool-friendly subset: {len(tool_dataset)} samples")

random.seed(RANDOM_SEED)
random.shuffle(tool_dataset)

split_point = int(len(tool_dataset) * 0.05)
DEV_SET = tool_dataset[:split_point]
TEST_SET = tool_dataset[split_point:]

print(f"DEV_SET:  {len(DEV_SET)} samples")
print(f"TEST_SET: {len(TEST_SET)} samples")

Tool-friendly subset: 365 samples
DEV_SET:  18 samples
TEST_SET: 347 samples


## 6. Scoring Mathematical Answers

For the toy problems we used `match(numeric=True)`, which works when answers are plain
numbers. But MATH answers can be fractions (`3/7`), expressions (`2\sqrt{5}`), or
formatted in different equivalent ways — a simple string match will miss many correct answers.

We covered `model_graded_qa()` in notebook 3. Here we apply it to math: the key challenge
is writing a grading prompt that handles equivalent notations (e.g. `1/2` vs `0.5` vs `\frac{1}{2}`).
Note that in notebook 3 we deliberately hid the reference answer from the grading model — here
there's no reason to do that, so you can pass the correct answer directly.

> **Trade-off:** Model-graded scoring is slower and noisier, but catches equivalences that
> string matching misses. For production evals you might want a more capable model for
> grading than the one being tested — think about what makes sense for your setup.

## Assignment 3: Define the math scorer

Define a `math_scorer` using `model_graded_qa()`. Write a grading prompt that instructs
the model to judge mathematical equivalence and respond with **C** (correct) or **I** (incorrect),
and choose which model should do the grading. Think about what edge cases matter: different
notations, equivalent fractions, simplified vs unsimplified forms.

In [22]:
from dotenv import load_dotenv
from inspect_ai.scorer import model_graded_qa

load_dotenv()
SCORER_MODEL = "openai/research-model"  # GLM-5.1 by Zhipu AI (open-weight)

GRADING_INSTRUCTIONS = """
Answer is correct, if it mathematically equal to correct answer.
Answer can be in different notations (1/2, \\frac{1}{2}, 0.5, 0,5).
Answer can be equivalent fractions or simplified/unsimplified form (7/14 is equivalent to 1/2, 13/2 is equivalent to 6.5, 2/8 is equivalent to 0.25).

Check submission. 
Print "yes" on the new line if answer is correct. 
Print "no" on the new line if answer is incorrect.

example:
yes

example:
no
"""

MATH_SCORER = model_graded_qa(
            #template=BLIND_TEMPLATE,
            instructions=GRADING_INSTRUCTIONS,
            model=SCORER_MODEL,
            grade_pattern=r"^.{,3}(yes|no).{,3}$",
)

In [23]:
# Run the scorer on two toy samples where we know the answer.
_scorer_test_samples = [
    Sample(input="What is 1+1?", target="2"),
    Sample(input="What is 10/4?", target="5/2"),
]

_log_scorer_test = eval(
    Task(
        dataset=_scorer_test_samples,
        solver=[system_message("Answer the math question. Reply with just the answer."), generate()],
        scorer=MATH_SCORER,
    ),
    model=MODEL,
)[0]

print_results("Scorer sanity check", _log_scorer_test)
print("Check: does the scorer correctly mark equivalent answers as C?")

Output()

  Scorer sanity check   accuracy: 100%
  [yes] # 1  msgs= 3  target=                   2  yes
  [yes] # 2  msgs= 3  target=                 5/2  yes

Check: does the scorer correctly mark equivalent answers as C?


---
1. Did your scorer correctly handle the fraction equivalence (`10/4` vs `5/2`)?
2. What failure modes can you imagine for model-graded scoring?

**Your answer:**
1. It handles `2.5 ~ 5/2`, yes.
2. ...
    - model is too weak to check equivalency (false positive or false negatives).
        - Can be cured with CoT or stronger models.
    - failure on correct but long answers - e.g. 12121212121212121212 2424242424242424242424/2 or smth even more difficult like 2096969696969696969676/173.
        - Maybe add tools to checker model?
    - следующие должны лечиться улучшением промта, если будут актуальны
        - ложноположительное срабатывание если ответ близкий численно но некорректный.
        - ложноположительные срабатывания на ответах вида `22*33+55` (значение выражения корректный ответ, но это выражение, а не ответ).
        - необычные форматы ответа могут быть не распознаны (например, текстовый ввод `двадцать три`).
    - (almost impossible for strong models) it can misformat final answer (yes/no)
    - (дороговизна)

## 7. Dev-Set Iteration — Building and Improving Your Agent

This is the core of agent evaluation: run on the dev set, see where the agent struggles,
and systematically improve it. The iteration loop:

1. Run the current agent on the dev set
2. Inspect failures — *why* did the agent get these wrong?
3. Hypothesize an improvement (better prompt? more tools? output formatting?) — and check
   whether the answer was actually correct but the scorer marked it wrong
4. Implement the change and re-evaluate on the dev set
5. Repeat

## Assignment 4: Build and improve a ReAct agent

Your goal is to build a ReAct agent and iterate on it using the dev set.
Does adding tools and scaffolding help on this task, and by how much?
Start by running a basic ReAct agent on the dev set, then look at the failures in the logs and iterate.

**Step 1 — Run a baseline.** Write a system prompt and run `react()` with `ALL_TOOLS` on the dev set.

**Step 2 — Inspect failures.** Open the logs and look at what went wrong. Common things to consider:
- Is the model ignoring tools and is failing?
- Is the answer mathematically correct but formatted wrong (e.g. `0.5` instead of `1/2`)?
- Is the model getting lost in multi-step problems?

**Step 3 — Iterate.** Based on what you see, try to improve. Some directions:
- Make the system prompt more explicit about strategy or answer format
- Add tools that cover operations the model struggles with (e.g. a single `calculator` tool, `gcd`, `factorial` or something very different)
- Add a format-extraction step after the `react()` loop if formatting is the main issue
- Reconsider the scorer if correct answers are being marked wrong

For each configuration you try, store the result as a `(description, log)` tuple in `DEV_RUNS` at the bottom — this will let you compare all your attempts in one table. Try at least two configurations.

In [ ]:
MY_REACT_PROMPT = """
You have tools. Use them to solve the task, then submit the result.
"""

# basic react prompt

log_attempt_1 = eval(
    Task(
        dataset=DEV_SET,
        solver=react(
            prompt=MY_REACT_PROMPT,
            tools=ALL_TOOLS,
            attempts=1,
        ),
        scorer=MATH_SCORER,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
)[0]

print_results("Attempt 1 (baseline)", log_attempt_1)
DEV_RUNS = [("Attempt 1 (baseline)", log_attempt_1)]

In [27]:
MY_REACT_PROMPT_2 = """You are a precise math-solving agent. You solve problems step by step using tools.

## Rules
1. Think before each action. Break complex problems into small steps.
2. Use EXACTLY ONE tool per step. Never skip tools for calculations.
3. Never compute arithmetic in your head — always use a tool.
4. After getting a tool result, reason about what to do next.
5. When done, give the final answer clearly with tool .

## Available Tools

### add(a, b)
Returns a + b.

### subtract(a, b)
Returns a - b. Use for subtraction: subtract(a: 10, b: 3) → 7.

### multiply(a, b)
Returns a × b.

### divide(a, b)
Returns a ÷ b as a decimal. Example: divide(a: 7, b: 2) → 3.5.

### modular_arithmetic(a, b)
Returns remainder of a ÷ b.
Example: modular_arithmetic(a: 17, b: 5) → 2

### sympy_solve(equation)
Solves an equation for x. Pass equation as a string with "=" sign.
Example: sympy_solve(equation: "2*x + 6 = 20") → [7.0]
Example: sympy_solve(equation: "x**2 - 9 = 0") → [-3.0, 3.0]

### submit(answer)
Submit final answer. Pass the answer as a string.
Example: submit(answer: "3")

## Strategy
- For multi-step arithmetic: do ONE operation at a time, use the result in the next call.
- For equations: write the equation as a string and call sympy_solve.
- For division with remainder: use modular_arithmetic, not divide.
- Always verify your answer makes sense before responding.

## Examples

**Problem:** What is (12 + 8) × 3?
**Steps:**
1. add(a: 12, b: 8) → 20
2. multiply(a: 20, b: 3) → 60
3. submit(answer: "60")

**Problem:** Solve 5*x - 15 = 0
**Steps:**
1. sympy_solve(equation: "5*x - 15 = 0") → 3.0
2. submit(answer: "3.0")

**Problem:** remainder of 29 divided by 6?
**Steps:**
1. modular_arithmetic(a: 29, b: 6) → 5
2. submit(answer: "5")"""

In [29]:
GRADING_INSTRUCTIONS_STANDARD = """
Answer is correct, if it mathematically equal to correct answer.
Answer can be in different notations (1/2, \\frac{1}{2}, 0.5, 0,5).
Answer can be equivalent fractions or simplified/unsimplified form (7/14 is equivalent to 1/2, 13/2 is equivalent to 6.5, 2/8 is equivalent to 0.25).

Check submission. 
Print "GRADE: C" on the new line if answer is correct. 
Print "GRADE: I" on the new line if answer is incorrect.

example:
GRADE: C

example:
GRADE: I
"""


@tool
def sympy_solve():
    async def execute(equation: str) -> str:
        '''Solve the equation for x.
        Args:
            equation: equation for x. * form multiply, ** for power. contains exactly 1 equal sign

        Returns:
            comma separated solutions

        Examples:
            >>> sympy_solve('2x=4')
            2
            >>> sympy_solve('x**2-5*x+3=3')
            2, 3
            >>> sympy_solve('500*x+10=60-500*x')
            0.05
        '''
        if not isinstance(equation, str):
            return "Error: equation is not string."
        eqs = equation.split('=')
        if len(eqs) <= 1:
            return "Error: equation must contain exactly 1 equal sign"
        elif len(eqs) >= 3:
            return "Error: equation must contain exactly 1 equal sign"
        try:
            eq0_sympy = parse_expr(eqs[0], transformations=_transformations)
        except Exception:
            return 'Error: incorrect equation'

        try:
            eq1_sympy = parse_expr(eqs[1], transformations=_transformations)
        except Exception:
            return 'Error: incorrect equation'
        
        result_sympy = eq0_sympy - eq1_sympy
        try:
            solutions = solve(result_sympy)
        except Exception:
            return "Error: can't solve"
        
        if not isinstance(solutions, list):
            return str(solutions)
        elif len(solutions) == 0:
            return "No solutions."
        else:
            return ', '.join(str(s) for s in solutions)
            
    return execute

ARITH_TOOLS_2 = [add(), subtract(), multiply(), divide(), modular_arithmetic()]
ALL_TOOLS_2 = ARITH_TOOLS_2 + [sympy_solve()]


MATH_SCORER_STANDARD = model_graded_qa(
            instructions=GRADING_INSTRUCTIONS_STANDARD,
            model=SCORER_MODEL,
            grade_pattern=r"(?is)(?:^|\n)\s*(?:GRADE\s*:\s*)?(C|I)\b",
)

log_attempt_2 = eval(
    Task(
        dataset=DEV_SET,
        solver=react(
            prompt=MY_REACT_PROMPT_2,
            tools=ALL_TOOLS_2,
            attempts=1,
        ),
        scorer=MATH_SCORER_STANDARD,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
)[0]

description_2 = "Improved prompt with LLM, and description of sympy_solver tool, also changed scorer to 'standard'. Hope it'll be much better"
print_results(description_2, log_attempt_2)
DEV_RUNS.append((description_2, log_attempt_2))

Output()

  Improved prompt with LLM, and description of sympy_solver tool, also changed scorer to 'standard'. Hope it'll be much better   accuracy: 17%
  [I] # 1  msgs=13  target=                  10  GRADE: I
  [I] # 2  msgs= 9  target=                  47  GRADE: I
  [I] # 3  msgs=16  target=                2k+2  GRADE: I
  [I] # 4  msgs= 5  target=                   2  GRADE: I
  [I] # 5  msgs= 5  target=                   5  GRADE: I
  [I] # 6  msgs=10  target=          (a+5)(b+2)  GRADE: I
  [C] # 7  msgs=17  target=                 550  GRADE: C
  [I] # 8  msgs=12  target=                   2  GRADE: I
  [C] # 9  msgs= 9  target=                   0  The problem asks for the sum $a+b$ given the domain of the f
  [I] #10  msgs=19  target=                   1  GRADE: I
  [C] #11  msgs=13  target=                   3  GRADE: C
  [I] #12  msgs=15  target=         \frac{3}{2}  GRADE: I
  [I] #13  msgs=20  target=                   1  The problem asks for the value of $f(2015, 2)$ for a recursi

In [ ]:
MY_REACT_PROMPT_3 = """You are a precise math-solving agent. You solve problems step by step using tools.

## Rules
1. Think before each action. Break complex problems into small steps.
2. Use EXACTLY ONE tool per step. Never skip tools for calculations.
3. Never compute arithmetic in your head — always use a tool.

## Available Tools

- add(a, b), subtract(a, b), multiply(a, b), divide(a, b)
- modular_arithmetic(a, b)
- sympy_solve(equation)
- submit(answer)

## Examples

**Problem:** What is (12 + 8) × 3?
**Steps:**
1. add(a: 12, b: 8) → 20
2. multiply(a: 20, b: 3) → 60
3. submit(answer: "60")

**Problem:** Solve 5*x - 15 = 0
**Steps:**
1. sympy_solve(equation: "5*x - 15 = 0") → 3.0
2. submit(answer: "3.0")

**Problem:** If a ? b = a*b + 3*b, calculate 3 ? 5
**Steps:**
1. multiply(a: 3, b: 5) -> 15
2. multiply(a: 3, b: 5) -> 15
3. add(a: 15, b: 15) -> 30
3. submit(answer: "30")

**Problem:** Simplify k + 2 + 3*k - 5 + 7 - 2*k
**Steps:**
1. add(a: 1, b: 3) -> 4  # k
2. substract(a: 4, b: 2) -> 2  # k
3. substract(a: 2, b: -5) -> -3
4. add(a: -3, b: 7) -> 4
5. submit(answer: "2*k + 4")
"""

GRADING_INSTRUCTIONS_STANDARD_3 = """
Answer is correct, if it mathematically equal to correct answer.
Answer can be in different notations (1/2, \\frac{1}{2}, 0.5, 0,5).
Answer can be equivalent fractions or simplified/unsimplified form (7/14 is equivalent to 1/2, 13/2 is equivalent to 6.5, 2/8 is equivalent to 0.25).

Check submission. 
Print "GRADE: C" on the new line if answer is correct. 
Print "GRADE: I" on the new line if answer is incorrect.

example:
GRADE: C

example:
GRADE: I
"""


@tool
def sympy_solve():
    async def execute(equation: str) -> str:
        '''Solve the equation for x.
        Args:
            equation: equation for x. * form multiply, ** for power. contains exactly 1 equal sign and only one unknown x

        Returns:
            comma separated solutions

        Examples:
            >>> sympy_solve('2x=4')
            2
            >>> sympy_solve('x**2-5*x+3=3')
            2, 3
            >>> sympy_solve('500*x+10=60-500*x')
            0.05
        '''
        if not isinstance(equation, str):
            return "Error: equation is not string."
        eqs = equation.split('=')
        if len(eqs) <= 1:
            return "Error: equation must contain exactly 1 equal sign"
        elif len(eqs) >= 3:
            return "Error: equation must contain exactly 1 equal sign"
        try:
            eq0_sympy = parse_expr(eqs[0], transformations=_transformations)
        except Exception:
            return 'Error: incorrect equation'

        try:
            eq1_sympy = parse_expr(eqs[1], transformations=_transformations)
        except Exception:
            return 'Error: incorrect equation'
        
        result_sympy = eq0_sympy - eq1_sympy
        try:
            solutions = solve(result_sympy)
        except Exception:
            return "Error: can't solve"
        
        if not isinstance(solutions, list):
            return str(solutions)
        elif len(solutions) == 0:
            return "No solutions."
        else:
            return ', '.join(str(s) for s in solutions)
            
    return execute

ARITH_TOOLS_3 = [add(), subtract(), multiply(), divide(), modular_arithmetic()]
ALL_TOOLS_3 = ARITH_TOOLS_3 + [sympy_solve()]


MATH_SCORER_STANDARD_3 = model_graded_qa(
            instructions=GRADING_INSTRUCTIONS_STANDARD_3,
            model=SCORER_MODEL,
            grade_pattern=r"(?is)(?:^|\n)\s*(?:GRADE\s*:\s*)?(C|I)\b",
)

log_attempt_3 = eval(
    Task(
        dataset=DEV_SET,
        solver=react(
            prompt=MY_REACT_PROMPT_3,
            tools=ALL_TOOLS_3,
            attempts=1,
        ),
        scorer=MATH_SCORER_STANDARD_3,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
)[0]

description_3 = "The same as with 2, but shorten the big react prompt, improved prompt for solver."
print_results(description_3, log_attempt_3)
DEV_RUNS.append((description_3, log_attempt_3))

In [86]:
!python --version

Python 3.11.1


In [69]:
MY_REACT_PROMPT_4 = MY_REACT_PROMPT_3
GRADING_INSTRUCTIONS_STANDARD_4 = GRADING_INSTRUCTIONS_STANDARD_3

# fixed sympy solve with sympy.sympify, also allowed no = sign - so it can use smth like r^2-4 as input
@tool
def sympy_solve():
    async def execute(equation: str) -> str:
        '''Solve the equation for one variable.
        Args:
            equation: equation for one variable (x, y or other one). * form multiply, ** for power, ^  also power.

        Returns:
            comma separated solutions

        Examples:
            >>> sympy_solve('2*x=4')
            2
            >>> sympy_solve('2*x-4')
            2
            >>> sympy_solve('r**2-5*r+3=3')
            2, 3
            >>> sympy_solve('500*x+10=60-500*x')
            0.05
        '''
        if not isinstance(equation, str):
            return "Error: equation is not string."

        eqs = equation.split('=')
        if len(eqs) >= 3:
            return "Error: equation must contain not more than 1 equal sign"
        elif len(eqs) == 0:
            return "Error: empty equation"
        
        if len(eqs) == 1:
            try:
                result_sympy = sympify(eqs[0])
            except Exception:
                return 'Error: incorrect equation'
        else:  # len(eqs) == 2:
            try:
                result_sympy = sympify(eqs[0] + f'-({eqs[1]})')
            except Exception:
                return 'Error: incorrect equation'
            
        
        try:
            solutions = solve(result_sympy)
        except Exception:
            return "Error: can't solve"
        
        if not isinstance(solutions, list):
            return str(solutions)
        elif len(solutions) == 0:
            return "No solutions."
        else:
            return ', '.join(str(s) for s in solutions)
            
    return execute

ARITH_TOOLS_4 = [add(), subtract(), multiply(), divide(), modular_arithmetic()]
ALL_TOOLS_4 = ARITH_TOOLS_4 + [sympy_solve()]


MATH_SCORER_STANDARD_4 = model_graded_qa(
            instructions=GRADING_INSTRUCTIONS_STANDARD_4,
            model=SCORER_MODEL,
            grade_pattern=r"(?is)(?:^|\n)\s*(?:GRADE\s*:\s*)?(C|I)\b",
)

In [ ]:

log_attempt_4 = eval(
    Task(
        dataset=DEV_SET,
        solver=react(
            prompt=MY_REACT_PROMPT_4,
            tools=ALL_TOOLS_4,
            attempts=1,
        ),
        scorer=MATH_SCORER_STANDARD_4,
        message_limit=MAX_MESSAGES,
    ),
    model=MODEL,
    limit=EVAL_LIMIT,
)[0]

description_4 = "The same as 3, but improved sympy tool - it can solve r^2-4 now"
print_results(description_4, log_attempt_4)
DEV_RUNS.append((description_4, log_attempt_4))

In [76]:
print(f"{'Configuration':<80s} {'Dev Accuracy':>12s}")
print("=" * 54)
for description, log in DEV_RUNS:
    acc = get_acc(log)
    print(f"{description:<80s} {acc:>12.0%}")

Configuration                                                                    Dev Accuracy
Attempt 1 (baseline)                                                                      28%
Improved prompt with LLM, and description of sympy_solver tool, also changed scorer to 'standard'. Hope it'll be much better          17%
The same as with 2, but shorten the big react prompt, improved prompt for solver.          39%
The same as 3, but improved sympy tool - it can solve r^2-4 now                           33%


---
1. What modifications did you try? Which had the biggest impact?
2. What was the best dev-set accuracy you achieved? What configuration produced it?
3. Did any change that you expected to help actually hurt? Why might that be?
4. Look at the individual failures. What are the most common error types?

**Your answer:**

1. Я пробовал менять промт ReAct агента, это повлияло сильнее всего. Также попробовал улучшить тул, оно не очень хорошее
2. 39% (3-й запуск). 4-й запуск хуже на 1 ответ, **выбираю его как лучший**.
3. Сначала я сделал большой промт, продублировал в нём также информацию об утилитах. Это работало ПЛОХО (17%), хотя я ожидал улучшения/неухудшения. Тогда я сократил этот промт, и добавил пример одной задачи из трейна.
4. Провалы (наблюдались >= 1 раза):
    - (>1 раза) Оно натыкается на sympy (форматирование выражения, требование знака равенства, требование переменной x). Это можно улучшить, улучшив эту утилиту.
    - (>1 раза) ошибочно вызывает sympy солвер, когда он не нужен
    - слишком рано останавливается, использует меньше тул чем нужно.
    - Делает ошибки при подсчёте символов.
    - Считает "не то", если задача упоминает другую задачу. Иногда ошибочно считает, что посчитал "не то".
    - Системы уравнений не может решать.

## 8. Test-Set Evaluation

You've iterated on the dev set and found your best configuration. Now it's time for the
moment of truth: evaluating on the held-out test set.

> **Run this section only once**, with your best configuration. Re-running and picking
> the best result would be "test-set contamination" — the same as peeking at a test set
> in ML.

## Assignment 5: Run and analyze your best configuration on the test set

Run your best agent configuration on the held-out test set, report accuracy with a 95%
confidence interval, and break down performance by subject and difficulty level. Note
anything that stands out.

In [97]:
def wilson_ci(n_correct, n_total, confidence_level=0.95):
    """Wilson score interval for a binomial proportion."""
    z = norm.ppf(0.5 + confidence_level / 2)
    p_hat = n_correct / n_total
    denom = 1 + z ** 2 / n_total
    centre = (p_hat + z ** 2 / (2 * n_total)) / denom
    margin = z * math.sqrt(
        (p_hat * (1 - p_hat) + z ** 2 / (4 * n_total)) / n_total
    ) / denom
    return max(0, centre - margin), min(1, centre + margin)

## Assignment 5.1: Run on the test set

Use your best configuration and run it to evaluate the test set

In [82]:
# shorten test_set to evaluate it

tool_dataset = [s for s in full_dataset if s.metadata["subject"] in TOOL_SUBJECTS]
print(f"Tool-friendly subset: {len(tool_dataset)} samples")

random.seed(RANDOM_SEED)
random.shuffle(tool_dataset)

split_point = int(len(tool_dataset) * 0.05)
split_point_2 = int(len(tool_dataset)*0.4)
DEV_SET = tool_dataset[:split_point]
TEST_SET = tool_dataset[split_point:split_point_2]

print(f"DEV_SET:  {len(DEV_SET)} samples")
print(f"TEST_SET: {len(TEST_SET)} samples")

Tool-friendly subset: 365 samples
DEV_SET:  18 samples
TEST_SET: 128 samples


In [91]:
from inspect_ai import task
@task
def final_task():
    return Task(
        dataset=TEST_SET,
        solver=react(
            prompt=MY_REACT_PROMPT_4,
            tools=ALL_TOOLS_4,
            attempts=1,
        ),
        scorer=MATH_SCORER_STANDARD_4,
        message_limit=MAX_MESSAGES,
    )

In [ ]:
log_test = eval(
    final_task(),
    max_connections=1,
    timeout=900,
    attempt_timeout=900,
    model=MODEL,
    limit=len(TEST_SET),
)[0]


In [95]:
# from inspect_ai import eval_retry
# log_test = eval_retry("logs/2026-04-12T14-38-39-00-00_final-task_9NsrsYZFUC6smq2v4ZEvBw.eval")[0]

[04/12/26 23:16:22] ERROR    Exception in callback Task.__step()                                ]8;id=644055;file:///home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/base_events.py\base_events.py]8;;\:]8;id=973611;file:///home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/base_events.py#1765\1765]8;;\
                             handle: <Handle Task.__step()>                                                        
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "/home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/events.p                    
                             y", line 80, in _run                                                                  
                                 self._context.run(self._callback, *self._args)                                    
                             RuntimeError: cannot enter context: <_contextvars.Context object                      
                             at 0x7f1894797040> is already entered                                                 

                    ERROR    Exception in callback Task.__step()                                ]8;id=576786;file:///home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/base_events.py\base_events.py]8;;\:]8;id=321802;file:///home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/base_events.py#1765\1765]8;;\
                             handle: <Handle Task.__step()>                                                        
                             Traceback (most recent call last):                                                    
                               File                                                                                
                             "/home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/events.p                    
                             y", line 80, in _run                                                                  
                                 self._context.run(self._callback, *self._args)                                    
                             RuntimeError: cannot enter context: <_contextvars.Context object                      
                             at 0x7f1894797040> is already entered                                                 

                    ERROR    Task was destroyed but it is pending!                              ]8;id=606799;file:///home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/base_events.py\base_events.py]8;;\:]8;id=618926;file:///home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/base_events.py#1765\1765]8;;\
                             task: <Task pending name='Task-374485'                                                
                             coro=<_async_in_context.<locals>.run_in_context() done, defined at                    
                             /home/scow/Desktop/evals/venv/lib/python3.11/site-packages/ipykern                    
                             el/utils.py:57> wait_for=<Task pending name='Task-374486'                             
                             coro=<Kernel.shell_main() running at                                                  
                             /home/scow/Desktop/evals/venv/lib/python3.11/site-packages/ipykern                    
                             el/kernelbase.py:597> cb=[Task.__wakeup()]>                                           
                             cb=[ZMQStream._run_callback.<locals>._log_error() at                                  
                             /home/scow/Desktop/evals/venv/lib/python3.11/site-packages/zmq/eve                    
                             ntloop/zmqstream.py:563]>                                                             

/home/scow/.pyenv/versions/3.11.1/lib/python3.11/json/decoder.py:353: RuntimeWarning: coroutine 'Kernel.shell_main' was never awaited
  obj, end = self.scan_once(s, idx)


                    ERROR    Task was destroyed but it is pending!                              ]8;id=810571;file:///home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/base_events.py\base_events.py]8;;\:]8;id=37849;file:///home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/base_events.py#1765\1765]8;;\
                             task: <Task pending name='Task-374486' coro=<Kernel.shell_main()                      
                             running at                                                                            
                             /home/scow/Desktop/evals/venv/lib/python3.11/site-packages/ipykern                    
                             el/kernelbase.py:597> cb=[Task.__wakeup()]>                                           

                    ERROR    Task was destroyed but it is pending!                              ]8;id=398423;file:///home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/base_events.py\base_events.py]8;;\:]8;id=935391;file:///home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/base_events.py#1765\1765]8;;\
                             task: <Task pending name='Task-374488'                                                
                             coro=<_async_in_context.<locals>.run_in_context() done, defined at                    
                             /home/scow/Desktop/evals/venv/lib/python3.11/site-packages/ipykern                    
                             el/utils.py:57> wait_for=<Task pending name='Task-374489'                             
                             coro=<Kernel.shell_main() running at                                                  
                             /home/scow/Desktop/evals/venv/lib/python3.11/site-packages/ipykern                    
                             el/kernelbase.py:597> cb=[Task.__wakeup()]>                                           
                             cb=[ZMQStream._run_callback.<locals>._log_error() at                                  
                             /home/scow/Desktop/evals/venv/lib/python3.11/site-packages/zmq/eve                    
                             ntloop/zmqstream.py:563]>                                                             

                    ERROR    Task was destroyed but it is pending!                              ]8;id=400056;file:///home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/base_events.py\base_events.py]8;;\:]8;id=849195;file:///home/scow/.pyenv/versions/3.11.1/lib/python3.11/asyncio/base_events.py#1765\1765]8;;\
                             task: <Task pending name='Task-374489' coro=<Kernel.shell_main()                      
                             running at                                                                            
                             /home/scow/Desktop/evals/venv/lib/python3.11/site-packages/ipykern                    
                             el/kernelbase.py:597> cb=[Task.__wakeup()]>                                           

Output()

In [98]:

n_test = len(TEST_SET)
n_correct_test = int(round(get_acc(log_test) *  n_test))

lo, hi = wilson_ci(n_correct_test, n_test)
print(f"Test accuracy : {n_correct_test / n_test:.1%}")
print(f"95% Wilson CI : [{lo:.1%}, {hi:.1%}]")
print(f"n = {n_test}")

Test accuracy : 32.0%
95% Wilson CI : [24.6%, 40.5%]
n = 128


## TEST_SET result

- Tool-friendly subset: 365 samples
- DEV_SET:  18 samples
- TEST_SET: 128 samples

---

- Test accuracy : 32.0%
- 95% Wilson CI : [24.6%, 40.5%]
- n = 128

## Assignment 5.2: Breakdown by subject and difficulty level

There aren't enough samples per category to draw firm conclusions — that's fine.
Just see if anything stands out.

Iterate over logs and produce **two separate tables**: one grouped
by subject, one by difficulty level. 

Your output should look like this:

**By subject:**

| Subject | Correct | Total | Acc |
|---|---:|---:|---:|
| Algebra | 1 | 10 | 10% |
| Precalculus | 3 | 6 | 50% |
| ... | | | |

**By level:**

| Level | Correct | Total | Acc |
|---|---:|---:|---:|
| 1 | 4 | 5 | 80% |
| 2 | 3 | 6 | 50% |
| ... | | | |

The actual numbers will appear in the cell below.

In [99]:
subject_stats = defaultdict(lambda: [0, 0])
level_stats = defaultdict(lambda: [0, 0])

for sample in log_test.samples:
    sc = _first_score(sample)
    correct = 1 if sc.value == "C" else 0
    subj = sample.metadata.get("subject", "unknown")
    lvl = sample.metadata.get("level", 0)
    subject_stats[subj][0] += correct
    subject_stats[subj][1] += 1
    level_stats[lvl][0] += correct
    level_stats[lvl][1] += 1

print(f"{'Subject':<30s} {'Correct':>7s} {'Total':>5s} {'Acc':>6s}")
print("-" * 50)
for subj in sorted(subject_stats):
    c, t = subject_stats[subj]
    print(f"{subj:<30s} {c:>7d} {t:>5d} {c/t:>6.0%}")

print()
print(f"{'Level':<30s} {'Correct':>7s} {'Total':>5s} {'Acc':>6s}")
print("-" * 50)
for lvl in sorted(level_stats):
    c, t = level_stats[lvl]
    print(f"Level {lvl:<24d} {c:>7d} {t:>5d} {c/t:>6.0%}")

Subject                        Correct Total    Acc
--------------------------------------------------
Algebra                             24    43    56%
Intermediate Algebra                 8    38    21%
Number Theory                        1    19     5%
Prealgebra                           8    28    29%

Level                          Correct Total    Acc
--------------------------------------------------
Level 1                              7     9    78%
Level 2                             15    23    65%
Level 3                             10    25    40%
Level 4                              6    36    17%
Level 5                              3    35     9%


**Your results:**

Fill in the tables once you've run the cell above.

**By subject:**
| Subject                       | Correct | Total | Acc |
|---|---|---|---|
| Algebra                            | 24  |  43  |  56%|
|Intermediate Algebra  | 8  |  38  | 21%|
|Number Theory                      | 1 |   19    | 5% |
|Prealgebra                         | 8 |  28  | 29% |

**By difficulty level:**

|Level |                         Correct | Total |   Acc|
|---|---|---|---|
| 1 |                             7  |   9  |  78% |
| 2 |                            15  |  23  |  65% |
| 3 |                            10  |  25  |  40% |
| 4 |                             6  |  36  |  17% |
|5 |                             3  |  35  |   9% |



## Final comparison: dev vs test

In [100]:
print(f"{'Configuration':<30s} {'Dev Acc':>8s}  {'Test Acc':>8s}")
print("=" * 50)
for name, log in DEV_RUNS:
    acc = get_acc(log)
    print(f"{name:<30s} {acc:>8.0%}  {'--':>8s}")
print(f"{'best agent (TEST)':<30s} {'--':>8s}  {n_correct_test / n_test:>8.1%}")
print(f"\n95% CI on test: [{lo:.1%}, {hi:.1%}]")

Configuration                   Dev Acc  Test Acc
Attempt 1 (baseline)                28%        --
Improved prompt with LLM, and description of sympy_solver tool, also changed scorer to 'standard'. Hope it'll be much better      17%        --
The same as with 2, but shorten the big react prompt, improved prompt for solver.      39%        --
The same as 3, but improved sympy tool - it can solve r^2-4 now      33%        --
best agent (TEST)                    --     32.0%

95% CI on test: [24.6%, 40.5%]


---
1. How does the test accuracy compare to the dev accuracy? If there's a gap,
   what might explain it?
2. Which subjects and difficulty levels does the agent handle best? Worst?
3. Look at the confidence interval. Is it narrow enough to be useful, or would
   you want more test samples?
4. If you were to improve this agent further, where would you focus?

**Your answer:**

1. На dev лучший результат 39%, на test 32%. Малое количество примеров в test, возможно также наличие разных категорий/сложностей неравномерно попавших в dev.
2. Уровень сложности: точность падает от 1 до 5. Лучше всего: алгебра (56%), хуже всего - теория чисел (5%).
3. CI [24.6%, 40.5%], недостаточно узкий - я бы хотел больше samples (у меня было ~=128 из-за недостатка вычислительных ресурсов).
4. Возможные улучшения
    1) (для удобства и чтобы пайплайн не зависал) улучшил бы tool `sympy_solve`, добавил бы возможность прерываться, если запущен больше 60 секунд (один раз он завис на несколько часов).
    2) (улучшение tools) сделал бы tool `calculate` чтобы считать простые выражения, типа `calculate(123*2+50*4) (=446)` (т.к. нередки были ошибки при разбиении выражения на части и последующем подсчёте - например, останавливалось на промежуточном результате). Добавил бы tool `gcd` для отдельных задач.
    3) поэкспериментировал бы с температурой (2-3 эксперимента) на DEV.
    4) few-shot prompting на более сложные задачи, с учётом tool `calculate`.
    5) улучшение модели/дообучение на успешных решениях подобных задач, если есть такая возможность.

## Bonus assignment: Error Analysis

Pick 5-10 test samples that the agent got wrong and classify each failure. Here's one
possible taxonomy — feel free to use your own:

- **Tool misuse:** Called the wrong tool or with wrong arguments
- **Reasoning error:** Correct tool use but flawed multi-step reasoning
- **Format error:** Correct answer but wrong format in `submit()`
- **Inherent difficulty:** Problem requires reasoning beyond the tool set
- **Grading error:** The agent was actually right but the scorer got it wrong
- **Other:** Anything that doesn't fit the above

This kind of qualitative error analysis is essential in practice — aggregate metrics
tell you *how much* the agent struggles, but error analysis tells you *why*.

If you'd like to explore the logs in code, use the cell below — otherwise just read
them directly and delete it.

---
1. What was the most common failure mode?
2. Which failure modes could be fixed with better prompting vs. better tools
   vs. a more capable model?
3. Did you find any grading errors? What does that imply about using model-graded scoring?

**Your answer:**

1. Беру 10 последних incorrect из тест-сета. Каждый может включать > 1 ошибки.

- **Reasoning error:** 8
- **Tool misuse:** 5
- **Tool недостаточность:** 1
- **Exceding message limit:** 1
- **Other:** 1 "This implies that essentially no decrease has occurred as the overall area remains at 90% (or an equivalent percentage decrease from 100%) of the original square."
- **Format error:** 0
- **Grading error:** 0

2. по режимам провала
- `Tool misuse` может чиниться лучшим промтингом/лучшими тулами (которые будут сами чинить некорректные их вызовы/яснее сообщать ошибки)/лучшей моделью.
- `Reasoning error` чинятся лучшей моделью, возможно лучшим промтингом.
- `Tool недостаточность` чинится лучшими/дополнительными тулами.
- `Exceding message limit` частично чинится большим лимитом/лучшей моделью.
- `Other` (здесь являлось ошибкой вида "мы видим 0.9, что почти 1, поэтому 1") должен чинится лучшим промтом (запрещающим слишком сильно округлять), также лучшей моделью.
- `Format error` (не наблюдалось) - модель, также промт.
- `Grading error` (не наблюдалось) - лучшая оценивающая модель, лучше оценивающий промт.

3. Не нашёл ошибок grading, в данном случае с достаточно сильной моделью (`GLM-5.1 by Zhipu AI (open-weight)` которая от aicohort), на простых ответах (число/выражение) - оно хорошо работает. Можно провести более подробный анализ конкретно данной модели на разных форматах ответов.